# Final Check — Verify the Manipulation Task

Verify the two-cube receiving-box task for SO-101 and reBot. Both CPU runs execute by default. The earlier stack and coupled cloth-and-cable exercises remain optional and have separate acceptance criteria.

Choose references or your completed exercise files explicitly. CPU checks run by default; CUDA checks that do not run **remain unverified**. See the [verification record](../VERIFICATION.md) for the tested reference runs.

## Outline

- Confirm the kernel, working directory and reference/student mode.
- Inspect the selected programs and devices.
- Review the two-cube box checks, then the optional stack and coupled-task criteria.
- Run the CPU suite for both robots and inspect its results.
- Optionally request CUDA verification on a selected device.

## What you will learn

You will use the same task gate from notebooks, scripts and rendering tools, distinguish a completed pickup from an object merely ending in the box, and keep reference results separate from verification of your own exercises.

## Setup: use the Article 3 kernel

Run with **`tutorials/newton/notebooks/newton/part3` as the kernel's working directory**. Follow the [setup guide](../README.md) for the Python 3.12 environment. The optional `requirements.lock.txt` fixes package versions and hashes; `requirements.txt` contains the direct dependency constraints.

From the repository root, launch Jupyter with the OpenUSD Work limit set:

```bash
PXR_WORK_THREAD_LIMIT=1 tutorials/newton/.venv/bin/jupyter lab
```

Select the Article 3 interpreter. With usd-core 26.3, restart an existing kernel after changing the launch environment; a late assignment after `pxr` loads is insufficient. On PowerShell use `$env:PXR_WORK_THREAD_LIMIT="1"` before launching. This parser workaround follows native failures on macOS and Linux; Windows has not been validated.

`REFERENCE = True` selects `solutions/` directly. After finishing the external exercises, set it to `False`, restart, and rerun. The checker does not fill TODOs or turn an unfinished exercise into a skip. Its Article 2 checks use the current Article 3 interpreter, not the earlier article's dependency snapshot.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import shlex
import sys

PART3 = Path.cwd().resolve()
assert PART3.name == "part3" and (PART3 / "final_check.py").is_file(), (
    "Start this kernel in <repository>/tutorials/newton/notebooks/newton/part3."
)
REFERENCE = True  # False checks your completed student files, not solutions.
RUN_LEGACY_AND_COUPLED = False  # Enable after completing those separate exercises.
print("Python:", sys.executable)
print("Directory:", PART3)
print("Mode:", "reference solutions" if REFERENCE else "student exercises")
for package in ("newton", "warp-lang", "mujoco", "mujoco-warp", "newton-usd-schemas", "usd-core"):
    print(f"{package}: {version(package)}")
assert version("newton") == "1.6.0", "Select the Article 3 kernel."
from pxr import Usd, Work
print("OpenUSD:", Usd.GetVersion(), "Work concurrency:", Work.GetConcurrencyLimit())
if Usd.GetVersion() == (0, 26, 3):
    assert Work.GetConcurrencyLimit() == 1, "Restart Python/Jupyter with PXR_WORK_THREAD_LIMIT=1"

from final_check import checks_for, run_final_check

## Check the two-cube box task first

Run the primary rigid-body task for both robots, using the completed reference or your substep exercise. Each report must pass the physical gates and count all 40,000 integration steps. The existing stack and coupled-material checks below cover their separate exercises.


In [ ]:
from datetime import datetime, timezone
import json
import subprocess

BOX_OUTPUT = PART3 / ".generated" / "final_box_check" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
BOX_SCRIPT = PART3 / ("solutions/so101_newton_solution.py" if REFERENCE else "box_newton_exercise.py")
for robot in ("so101", "rebot"):
    report = BOX_OUTPUT / f"{robot}_cpu.json"
    subprocess.run([sys.executable, str(BOX_SCRIPT), "--task", "box", "--robot", robot,
                    "--device", "cpu", "--viewer", "null", "--num-frames", "2000",
                    "--sim-substeps", "20", "--test", "--report", str(report)], check=True, timeout=1800)
    result = json.loads(report.read_text())
    assert result["status"] == "passed" and result["capacity_passed"]
    assert result["validation"]["passed"] and result["recorded_physics_steps"] == 40000
print("Both CPU box tasks passed; CUDA is a separate check.")


## Overview of the files

| File | Role |
|---|---|
| `final_check.py` | Runs the selected programs and applies the pure stack/gripper report validators |
| `tutorials/newton/notebooks/mujoco/part1/so101_pick_place.py` | CPU MuJoCo stack exercise |
| `tutorials/newton/notebooks/mujoco/part2/so101_mjwarp.py` | One-world MJWarp stack exercise; CUDA-only in this suite |
| `tutorials/newton/notebooks/newton/part3/so101_newton.py` | Newton stack exercise, with a native CPU baseline |
| `tutorials/newton/notebooks/newton/part3/clean_the_table.py` | Sequential pickup of all four objects; selected by `include_clean_table=True` |
| `solutions/*_solution.py` | Reference counterparts selected by reference mode / `--solutions` |
| `.generated/final_check/` | Reports and trajectories written by these checks |

The advanced scene uses the official robot USD, MuJoCo robot/cube ownership and VBD cloth/rod ownership. The report validator is independent of physics imports and is shared with the offline renderer. It demands the current gripper-task schema rather than accepting any successful-looking JSON.

## Optional earlier stack and coupled-material checks

The primary box checks above run by default. Set `RUN_LEGACY_AND_COUPLED = True` to include the separate earlier stack and advanced cloth/cable exercises.

### Inspect the selected checks

The script name is not the whole configuration. Inspect the robot, viewer and physics options as well. `--skip-gpu` skips CUDA checks and forces the optional coupled task onto CPU; the baseline Newton stack check requests native CPU explicitly. `--include-newton-cuda` adds stack checks for both Newton-generated and MuJoCo-generated contacts on CUDA, and `--device cuda:N` selects the GPU for all GPU checks.

The following cell lists both robots. It does not run anything or label any check a pass.

In [ ]:
selected = [
    check
    for robot in ("so101", "rebot")
    for check in checks_for(robot, solutions=REFERENCE, include_clean_table=True)
]
for check in selected:
    print(check.name)
    print("  Script:", check.script)
    print("  Arguments:", shlex.join(check.args))
    print("  Metric:", check.metric, "| CUDA required:", check.requires_cuda, "| timeout:", check.timeout, "s")
assert selected and all(check.script.is_file() for check in selected), "A selected entry point is missing."

## Step 1. Know what counts as success

### The stack

The checker parses the last stack report and requires `0.0 <= xy_err <= 0.015 m` and `0.035 <= dz <= 0.055 m`. Both values must be finite. A missing/malformed report or a program that raises after printing plausible values fails.

### The gripper task

Under `--test`, the advanced program prints a JSON line beginning **`CLEAN_TABLE_RESULT `**. `validate_clean_table_report` requires `task="gripper_pick_place_into_bin"`, `schema_version=2`, the requested robot/device, positive `frames`, matching `simulation_seconds`, `phase="done"` and Boolean `success=true`.

Its `objects` map must contain exactly `red_cube`, `blue_cube`, `shirt` and `cable`. Every object must start outside the box and complete the following measured sequence:

| Stage | Required evidence |
|---|---|
| Grasp | At least 10 loaded bilateral-contact frames, with positive measured normal force from both jaws |
| Lift | At least 25 consecutive fully lifted frames; the lowest complete-object extent clears the table by at least 0.01 m |
| Carry | At least 10 loaded carry frames and 0.05 m of uninterrupted airborne XY movement; the whole object reaches above the bin |
| Open and release | `grasp_time < lift_time <= carry_time < release_command_time <= release_time`; measured opening fraction at least 0.8; no earlier dropped grasp |
| Detached settling | No jaw contacts, `inside=true`, maximum point speed below 0.04 m/s, and at least 50 consecutive `settled_frames`/`detached_settled_frames` |

The accepted object cycles must be sequential. Geometry includes cube corners, cloth particle radii and cable capsule extents, with a 3 mm containment tolerance. At the end, `tool_withdrawn=true` and finite `tool_clearance_m >= 0.02` establish clearance of the gripper above the table/rim reference. The report must also contain finite state-derived measurements, material contacts and nonzero coupling feedback; capacity overflow invalidates a run.

The current checked runner has a 5000-frame budget at 50 Hz and stops early only after measured completion. A target pose, phase name, eventual landing in the box or successful process exit alone does not establish a pickup.

Each advanced check saves a JSON/NPZ pair in `.generated/final_check/{reference|student}_clean_{robot}.{json,npz}`; explicit CUDA selection adds `_cuda0`, `_cuda1`, etc. The JSON's `record_sha256` binds the exact trajectory. Recordings include `phase` and `active_object` for inspecting the actual sequence. The renderer checks the same task contract and binding before exporting figures.

## Step 2. Optionally assert the earlier stack and coupled tasks

The coupled checks allow **1800 seconds per robot**, including initialization and the full sequence. This is a timeout budget, not a speed promise. If an object fails to complete its pickup, inspect the dynamics and event history rather than reducing the run to obtain a green cell.

The assertion is intentional: `run_final_check` returns `True` only when at least one check ran and every executed check passed. It reports skips separately. With `skip_gpu=True`, the result covers selected CPU tasks only; CUDA checks remain unverified.

When enabled, the coupled CPU cell runs both robots sequentially, with up to 30 minutes allowed for each coupled task. GPU elapsed times are not a prediction of CPU runtime.


In [ ]:
if RUN_LEGACY_AND_COUPLED:
    assert run_final_check(
        skip_gpu=True,
        robot="both",
        solutions=REFERENCE,
        include_clean_table=True,
    ), "An executed task failed. Read the named failure above; do not suppress this assertion."
else:
    print("Optional stack and coupled-material checks not requested.")


## Step 3. Read the result

| Status | Meaning |
|---|---|
| `PASS` | The selected program exited normally and its measured report met the gate |
| `FAIL` | A task, dependency, timeout or report contract failed; inspect the named output |
| `SKIP` | The check did not run and remains unverified |

A reference pass applies to the references in that environment. To check your own work, finish the external exercises, set `REFERENCE=False`, restart, and rerun. An unfinished starter should fail.

The suite does not benchmark throughput, GPU scaling or interactive rendering. Those need separate experiments even when both robot profiles pass the manipulation checks.

## Step 4. Optionally add CUDA checks

The optional run below requires a real CUDA device and checks the primary box task for both robots. If `RUN_LEGACY_AND_COUPLED` is also enabled, it additionally runs the earlier stack and coupled-material suite, including native CPU baselines and both Newton-generated and MuJoCo-generated contact paths. `GPU_DEVICE` selects one CUDA device. These correctness checks do not establish multi-GPU scaling.


In [ ]:
RUN_GPU_CHECKS = False
GPU_DEVICE = "cuda:0"  # or "cuda:1" on a host with a second visible GPU
if RUN_GPU_CHECKS:
    import warp as wp
    wp.init()
    assert wp.is_cuda_available(), "CUDA checks requested, but no CUDA device was found."
    assert wp.get_device(GPU_DEVICE).is_cuda, f"Requested device is not CUDA: {GPU_DEVICE}"
    for robot in ("so101", "rebot"):
        report = BOX_OUTPUT / f"{robot}_{GPU_DEVICE.replace(':', '')}.json"
        subprocess.run([sys.executable, str(BOX_SCRIPT), "--task", "box", "--robot", robot,
                        "--device", GPU_DEVICE, "--viewer", "null", "--report", str(report)],
                       check=True, timeout=1800)
        box_result = json.loads(report.read_text())
        assert box_result["status"] == "passed" and box_result["capacity_passed"]
        assert box_result["validation"]["passed"] and box_result["recorded_physics_steps"] == 40000
    if RUN_LEGACY_AND_COUPLED:
        assert run_final_check(
            skip_gpu=False, robot="both", solutions=REFERENCE, include_clean_table=True,
            include_newton_cuda=True, device=GPU_DEVICE,
        )
else:
    print("SKIP: CUDA checks not requested. They remain unverified; CPU results do not cover them.")

## Terminal and viewer checks

You can rerun the same suite from a terminal using the exact interpreter below. `--solutions` is added only in reference mode; omitting it is how you check your own files. A failed CLI suite returns a nonzero process exit status.

For visual diagnosis, run a single selected task with Newton's `gl` viewer on a local desktop. The interactive viewer is optional and is **not** part of the final assertion. If OpenGL is unavailable, keep using `--viewer null` rather than swallowing a task failure.

In [ ]:
mode_args = ["--solutions"] if REFERENCE else []
print("Final suite:")
print(shlex.join([sys.executable, str(PART3 / "final_check.py"), "--skip-gpu", "--robot", "both",
                  "--include-clean-table", *mode_args]))
print("Optional local clean-table viewer:")
clean_script = PART3 / ("solutions/clean_the_table_solution.py" if REFERENCE else "clean_the_table.py")
for robot in ("so101", "rebot"):
    print(shlex.join([sys.executable, str(clean_script), "--robot", robot,
                      "--device", "cpu", "--viewer", "gl", "--test"]))

## Troubleshooting

| Symptom | Next action |
|---|---|
| Student file raises `NotImplementedError` | Complete the named TODO, or deliberately select references. |
| Wrong package version | Check `sys.executable` and the Article 3 kernel; keep Article 2's environment separate. |
| USD Work-limit preflight fails | Restart Python/Jupyter with `PXR_WORK_THREAD_LIMIT=1` before importing `pxr`. |
| Cache validation fails | Preserve the existing cache; select a fresh cache location or a deliberate Menagerie path override as documented in the setup guide. |
| Report schema is rejected | Run the current gripper implementation; the gate requires task `gripper_pick_place_into_bin`, schema version 2. |
| Object ends inside but the task fails | Inspect bilateral loading, whole-object lift, carry continuity, opening and release events. |
| Object drops before opening | Fix the physical grasp/transport; later containment does not erase a lost grasp. |
| Object is placed but still moving | Inspect the detached settling interval and final jaw withdrawal. |
| Coupled run times out | Investigate runtime and task progress; no completed task was verified. |
| CUDA is skipped | Request a supported NVIDIA device when GPU verification is needed. |
| CUDA discovery or viewer fails | Fix that environment path; use a null viewer to isolate physics from rendering. |

## Recap

- Did the run select references or your completed exercises?
- Which robot, backend, contact path and device ran?
- What proves each payload was grasped and carried rather than dragged or dropped?
- Did both cubes detach and settle inside the receiving box? If the coupled exercise ran, did all four objects pass its separate checks?
- Which CUDA checks remain unverified?

## Next

Once a measured baseline passes, change one material or task parameter at a time and keep the acceptance checks. A training environment additionally needs observations, actions, resets and rewards. Use the [solver matrix](https://newton-physics.github.io/newton/stable/solvers/index.html) and [coupling guide](https://newton-physics.github.io/newton/stable/concepts/coupling.html) alongside the companion's pinned Newton source.